# D3 · Experiment 0 · sensor PSD (LR, SVM)

Dataset 3 (ds004504, AD vs HC): electrode band power (19 channels × 4 bands × {log, relative}).

**Runs only this experiment** (`configs/experiments/d3/d3_exp0_sensor_psd.yaml`): every predefined variant and setting, with subject-level cross-validation, evaluation and checkpointing. It never starts another experiment.

- **2 variant(s):** `logreg`, `svm`
- **Settings:** within_D3 (cross-dataset only if the DESIGN.md §7 gate passes)
- **Protocol (from the config, unchanged):** 20 repeats × 5-fold subject-level outer CV; inner 4-fold subject-level CV over 5 hyperparameter combination(s); refit on the full training fold.

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Progress** is shown as one updating line: completed outer folds of the whole experiment
(previously completed folds recovered from Drive count as done immediately), current variant,
setting, repeat and fold, inner-CV trial (if any), and the counts of finished /
resumed / failed / skipped units. An ETA is shown only for the current variant and setting, from folds
computed in this session.

Results are saved under `<outputs>/experiments_D3/d3_exp0_sensor_psd/` on Drive: per-fold/repeat checkpoints,
window and subject predictions, metrics per repeat, selected hyperparameters, summaries, manifests,
runtime and peak memory per unit (`_run_status.csv`), and a report folder (results table +
**single-experiment diagnostic figures**). The comparative paper figures across experiments are made
separately (`03_comparative_figures.ipynb`).

Requires the graphs from `d3/D3_01_download_and_build_graphs.ipynb`. Dataset 3 (OpenNeuro ds004504, AD vs HC): graph runs `graphsD3_v4_*`, results in `experiments_D3/`, never mixed with D1/D2.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only if you were asked to switch to newer code
                      # (all notebooks then use the newest commit; unfinished work is recomputed).
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)

## 0. Dataset 3 code pin
All Dataset 3 notebooks run one commit, recorded in `<outputs>/D3_CODE_COMMIT.txt` on the first D3 run; the
D1/D2 experiments keep their own pin (`CODE_COMMIT.txt`, unchanged). Leave `UPDATE_D3_CODE = False`.

In [ ]:
# Dataset 3 code pin: every D3 notebook (download, graphs, experiments) runs the commit recorded in
# <outputs>/D3_CODE_COMMIT.txt (set on the first D3 run), independent of the D1/D2 pin in CODE_COMMIT.txt.
UPDATE_D3_CODE = False   # leave False; True moves only the D3 pin to the newest code
import subprocess, sys
from pathlib import Path
from adgraph.colab import resolve_code_pin

head = subprocess.check_output(['git', '-C', '/content/repo', 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
d3_work = (any(Path(OUT).glob('graphsD3_v*/subjects/*/subject.done.json'))
           or any(Path(OUT).glob('experiments_D3/**/*.done.json')))
D3_COMMIT, _ = resolve_code_pin(Path(OUT) / 'D3_CODE_COMMIT.txt', head, update=UPDATE_D3_CODE, has_work=d3_work)
_run(['git', '-C', '/content/repo', 'checkout', '-q', D3_COMMIT])
for f in ('src/adgraph/io/bids.py', 'configs/pipeline_d3.yaml', 'scripts/d3_data.py'):
    if not Path('/content/repo', f).exists():
        raise RuntimeError(f'D3 pin {D3_COMMIT[:10]} lacks {f}: set UPDATE_D3_CODE = True and run again')
_run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/repo/requirements.txt'])
for m in [m for m in sys.modules if m == 'adgraph' or m.startswith('adgraph.')]:
    del sys.modules[m]                      # this notebook's own imports now come from the D3 commit
from adgraph.colab import choose_output_root, find_data_root, sh, write_paths_yaml
from adgraph.config import load_paths
_root = find_data_root()
write_paths_yaml('/content/repo', _root, choose_output_root(_root))   # adds the Dataset 3 location
PATHS = load_paths('/content/repo/configs/paths.yaml')
print('D3 code commit:', D3_COMMIT)
print('D1/D2 pin (unchanged):', (Path(OUT) / 'CODE_COMMIT.txt').read_text().split()[0])
print('Dataset 3 folder:', PATHS['dataset3']['root'])

## 1. This experiment: progress so far (from the checkpoints on Drive)

In [ ]:
CONFIG = 'configs/experiments/d3/d3_exp0_sensor_psd.yaml'   # this notebook's experiment
EXPERIMENT = 'd3_exp0_sensor_psd'
import torch
print('runtime:', ('GPU ' + torch.cuda.get_device_name(0)) if torch.cuda.is_available() else 'CPU only')
sh(f"python scripts/experiment.py status --config {CONFIG}")

## 2. Run (or resume) this experiment
Saves every completed fold to Drive as it goes. Units that fail are recorded and the others continue; failures are reported at the end of section 3 (after the results are shown) and stop *Run all* with an error.

In [ ]:
RUN_ERROR = None
try:
    sh(f"python scripts/experiment.py run --config {CONFIG}")
except RuntimeError as exc:
    RUN_ERROR = exc
    print('\nSome units failed or are incompatible; details after the results below.')

## 3. Results of this experiment (saved outputs only)
Subject-level metrics: mean ± SD over CV repeats, subject-bootstrap 95 % CI. Runtime and peak memory per variant and setting. Diagnostic figures of this experiment only; not the comparative paper figures. Nothing is selected or ranked here.

In [ ]:
sh(f"python scripts/experiment.py report --config {CONFIG}")
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
exp_dir = Path(OUT) / 'experiments_D3' / EXPERIMENT
report = sorted((exp_dir / '_reports').glob('report_*'))[-1]
print('report folder on Drive:', report)
pd.set_option('display.max_rows', 500, 'display.width', 250)
table = pd.read_csv(report / 'results_summary.csv')
display(table) if len(table) else print('no completed results yet')
status = pd.read_csv(exp_dir / '_run_status.csv') if (exp_dir / '_run_status.csv').exists() else pd.DataFrame()
if len(status):
    print('runtime and peak memory of the last run (seconds, MB):')
    display(status[['variant', 'setting', 'state', 'seconds', 'mem_peak_mb', 'message']])
for png in sorted((report / 'figures').glob('D[12]_*.png')):
    print(png.name)
    display(Image(filename=str(png)))
print('all figures:', sorted(p.name for p in (report / 'figures').glob('*.png')))
bad = status[status['state'].isin(['failed', 'incompatible'])] if len(status) else status
if RUN_ERROR is not None or len(bad):
    raise RuntimeError(f"{len(bad)} unit(s) of {EXPERIMENT} did not complete (see the table above and "
                       f"{exp_dir / '_run_status.csv'}); completed units are saved. Fix the cause, then Run all "
                       "again: only the incomplete units are recomputed.")